In [177]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import json
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import random

THRESHOLD_TIMESTAMPS = 16
L_SEQUENCE_LENGHT = 48

In [178]:
def extract_json(filename: str):
    with open(filename, "r") as f:
        for line in f:
            session = json.loads(line)
            yield session["session"], session["events"]

objects = extract_json("train.jsonl")

print(type(objects))


<class 'generator'>


In [179]:
sessions_bf_threshold = []

for i, (session_id, eventstotal) in enumerate(extract_json("../train.jsonl")):
    aids, timestamps, events_type = [], [], []
    for event in eventstotal:
        aids.append(event["aid"])
        timestamps.append(event["ts"])
        events_type.append(event["type"])
        
    sessions_bf_threshold.append({
            "session_id": i,
            "events": {
            "aid": aids,
            "timestamps": timestamps,
            "events_type": events_type    
            },
        })
    if i >= 200:
        break

In [180]:
class OttoDataSetSession(Dataset):
    def __init__(self, session):
        self.session = session
        self.event_map = {"clicks":1, "carts": 2, "orders": 3}

    def __len__(self) -> int:
        return len(self.session)

    def __getitem__(self, index):
        session = self.session[index]
                 
        events = session["events"]
        
        aids = torch.tensor(events["aid"], dtype=torch.int64)
        
        timestamps = torch.tensor(events["timestamps"], dtype=torch.long)
        
        events_type = torch.tensor( [self.event_map[e] for e in events["events_type"]], dtype=torch.int64)
        return {
            "session_id": torch.tensor(session["session_id"], dtype=torch.int64),
            "aid": aids,
            "timestamps": timestamps,
            "type": events_type
        }
    

In [215]:
sessions_in_dataset = OttoDataSetSession(sessions_bf_threshold)
print(f"Total len of the Sessions: {len(sessions_in_dataset)}")

session_sample_lenght_after_threshold = []
for i in range(len(sessions_in_dataset)):
    sample = sessions_in_dataset[i]
    if len(sample["timestamps"]) >= THRESHOLD_TIMESTAMPS:
        session_sample_lenght_after_threshold.append(sample)

#print(np.mean(session_sample_lenght_after_threshold))
#print(f"The total of the median is for this total of {len(session_sample_lenght_after_threshold)} {np.median(session_sample_lenght_after_threshold)}")

print(session_sample_lenght_after_threshold[0])

Total len of the Sessions: 201
{'session_id': tensor(0), 'aid': tensor([1517085, 1563459, 1309446,   16246, 1781822, 1152674, 1649869,  461689,
         305831,  461689,  362233, 1649869, 1649869,  984597, 1649869,  803544,
        1110941, 1190046, 1760685,  631008,  461689, 1190046, 1650637,  313546,
        1650637,  979517,  351157, 1062149, 1157384, 1841388, 1469630,  305831,
        1110548, 1110548,  305831, 1650114, 1604396, 1009750, 1800933,  495779,
         394655,  495779,  789245,  789245,  366890,  361317, 1700164, 1755597,
         789245,  784978, 1171505,  784978, 1700164,  784978, 1521766, 1725503,
         528847, 1816325,  984597, 1072782,  173702, 1072782, 1407538, 1629651,
        1768568, 1318324, 1840418, 1813509, 1813509,  667924, 1226444,  709550,
         709417, 1225559, 1048044, 1052813, 1225559,  240346, 1582117, 1707783,
        1624436, 1157411,  358305, 1202970,  832192, 1498443,  723931, 1436439,
        1693461, 1206554, 1110741,  346352, 1802050,  15

In [227]:
class CutOttoDataSet(OttoDataSetSession):
    def __init__(self, session):
        super().__init__(session)
        
        
    def __getitem__(self, index):
        session = self.session[index]

        return {
            "session_id": session["session_id"],
            "aid": session["aid"],
            "timestamps": session["timestamps"],
            "type": session["type"]
        }
        
    def __cut_training_tesing__(self, min_value=0.70,max_value=0.80):
        training_batches = []
        testing_batches = []
        for single_session in self.session:       
            cutting_number = random.uniform(min_value,max_value)
            number_training_index = int(len(single_session["timestamps"]) * cutting_number )
            number_testin_index = np.absolute(number_training_index - len(single_session["timestamps"]))
            training_batches.append(number_training_index)
            testing_batches.append(number_testin_index)
            
        return training_batches, testing_batches
    
    def __sequenceLenght__(self):
        return [L_SEQUENCE_LENGHT if i >= L_SEQUENCE_LENGHT else "Add_padding" for i in self.__cut_training_tesing__(0.70,0.80)[0]]


In [229]:
data_set_after_L = CutOttoDataSet(session_sample_lenght_after_threshold)
training, testing = data_set_after_L.__cut_training_tesing__()
print(f"Training Batch: {len(training)})")
print(f"Testing Batch: {len(testing)}")

print(data_set_after_L.__sequenceLenght__())


Training Batch: 154)
Testing Batch: 154
[48, 'Add_padding', 'Add_padding', 48, 'Add_padding', 48, 'Add_padding', 48, 48, 48, 48, 48, 'Add_padding', 48, 48, 48, 48, 48, 'Add_padding', 48, 48, 'Add_padding', 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 'Add_padding', 'Add_padding', 48, 48, 48, 'Add_padding', 48, 48, 'Add_padding', 48, 48, 'Add_padding', 'Add_padding', 'Add_padding', 'Add_padding', 48, 'Add_padding', 'Add_padding', 'Add_padding', 'Add_padding', 'Add_padding', 'Add_padding', 'Add_padding', 48, 'Add_padding', 48, 48, 'Add_padding', 'Add_padding', 48, 48, 48, 48, 48, 'Add_padding', 'Add_padding', 'Add_padding', 48, 48, 48, 48, 48, 'Add_padding', 48, 48, 'Add_padding', 'Add_padding', 'Add_padding', 'Add_padding', 48, 48, 48, 48, 'Add_padding', 'Add_padding', 48, 'Add_padding', 48, 'Add_padding', 'Add_padding', 'Add_padding', 'Add_padding', 48, 48, 'Add_padding', 'Add_padding', 48, 48, 'Add_padding', 'Add_padding', 48, 48, 48, 48, 'Add_padding', 48, 'Add_padding', 'Add_padd